Load the dataset

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np

X = np.load("/content/X_final.npy")
y = np.load("/content/y_final.npy")

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Label range:", y.min(), y.max())

Visual check random board

In [ ]:
i = np.random.randint(len(X))
print("Label:", y[i])
print(X[i,:,:,0] - X[i,:,:,1])

Convert to PyTorch dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class Connect4Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = Connect4Dataset(X, y)
print("Dataset size:", len(dataset))

Train/validation split

In [ ]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256)

Create CNN model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class Connect4CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 64, kernel_size=(3,3), padding='same')
        self.dropout1 = nn.Dropout(0.1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(3,3), padding='same')
        self.dropout2 = nn.Dropout(0.2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=(3,3), padding='same')
        self.dropout3 = nn.Dropout(0.3)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=(3,3), padding='same')
        self.dropout4 = nn.Dropout(0.4)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 1024)
        self.dropout_fc = nn.Dropout(0.3)
        self.fc2 = nn.Linear(1024, 7)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = F.relu(self.conv1(x))
        x = self.dropout1(x)
        x = F.relu(self.conv2(x))
        x = self.dropout2(x)
        x = F.relu(self.conv3(x))
        x = self.dropout3(x)
        x = F.relu(self.conv4(x))
        x = self.dropout4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return F.softmax(x, dim=1)

Create transformer model

In [ ]:
import tensorflow as tf
class PositionalIndex(tf.keras.layers.Layer):
    def call(self, x):
        bs = tf.shape(x)[0] # extract batch size
        number_of_vectors = tf.shape(x)[1] # how many vectors - we know it should be m*n, but let's count to be sure...
        indices = tf.range(number_of_vectors) # index for each vector
        indices = tf.expand_dims(indices, 0) # reshape it appropriately
        return tf.tile(indices, [bs, 1]) # repeat for each batch

In [ ]:
class ClassTokenIndex(tf.keras.layers.Layer):
    def call(self, x):
        bs = tf.shape(x)[0] # extract batch size
        number_of_vectors = 1 # how many vectors - we just want 1 (which is @ 0) ... we only want to generate 1 vector for the class token
        # now just get it to be the right size
        indices = tf.range(number_of_vectors) # index for each vector
        indices = tf.expand_dims(indices, 0) # reshape it appropriately
        return tf.tile(indices, [bs, 1]) # repeat for each batch

In [ ]:
def build_ViT(n,m,block_size,hidden_dim,num_layers,num_heads,key_dim,value_dim,mlp_dim,dropout_rate,num_classes):
    # n is number of rows of blocks
    # m is number of cols of blocks
    # block_size is number of pixels (with rgb) in each block
    inp = tf.keras.layers.Input(shape=(n*m,block_size))
    mid = tf.keras.layers.Dense(hidden_dim, use_bias=False)(inp) # transform to vectors with different dimension
    # the positional embeddings
    inp2 = PositionalIndex()(inp)
    emb = tf.keras.layers.Embedding(input_dim=n*m, output_dim=hidden_dim)(inp2) # learned positional embedding for each of the n*m possible possitions
    mid = tf.keras.layers.Add()([mid, emb])
    # create and append class token to beginning of all input vectors
    tokenInd = ClassTokenIndex()(mid)
    token = tf.keras.layers.Embedding(input_dim=1, output_dim=hidden_dim)(tokenInd)
    mid = tf.keras.layers.Concatenate(axis=1)([token, mid])

    for l in range(num_layers): # how many Transformer Head layers are there?
        ln  = tf.keras.layers.LayerNormalization()(mid) # normalize
        mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=key_dim,value_dim=value_dim)(ln,ln,ln) # self attention!
        add = tf.keras.layers.Add()([mid,mha]) # add and norm
        ln  = tf.keras.layers.LayerNormalization()(add)
        den = tf.keras.layers.Dense(mlp_dim,activation='relu')(ln) # maybe should be relu...who knows...
        den = tf.keras.layers.Dropout(dropout_rate)(den) # regularization
        den = tf.keras.layers.Dense(hidden_dim)(den) # back to the right dimensional space
        den = tf.keras.layers.Dropout(dropout_rate)(den)
        mid = tf.keras.layers.Add()([den,add]) # add and norm again

    fl = mid[:,0,:] # just grab the class token for each image in batch
    ln = tf.keras.layers.LayerNormalization()(fl)
    clas = tf.keras.layers.Dense(num_classes,activation='softmax')(ln) # probability that the image is in each category
    mod = tf.keras.models.Model(inp,clas)
    mod.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return mod

In [ ]:
n = 6
m = 7
block_size = 2
hidden_dim = 64
num_layers = 3
num_heads = 4
key_dim = hidden_dim//num_heads # usually good practice for key_dim to be hidden_dim//num_heads...this is why we do Multi-Head attention
value_dim = key_dim
mlp_dim = hidden_dim
dropout_rate = 0.2
num_classes = 7

trans = build_ViT(n,m,block_size,hidden_dim,num_layers,num_heads,key_dim,value_dim,mlp_dim,dropout_rate,num_classes)
trans.summary()

In [ ]:
X_vit = X.reshape(len(X), 6*7, 2).astype(np.float32)

Train models

In [ ]:
def train_model(model, train_loader, val_loader, epochs):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for i, (Xb, yb) in enumerate(train_loader):
            Xb, yb = Xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if i == 0:
                print(f"Epoch {epoch+1} started...")

        # validation
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        print(
            f"Epoch {epoch+1:02d} | "
            f"loss={total_loss:.2f} | "
            f"val_acc={correct/total:.3f}"
        )

In [ ]:
cnn = Connect4CNN()
train_model(cnn, train_loader, val_loader, epochs=25)
torch.save(cnn.state_dict(), "cnn_connect4.pt")

Epoch 1 started...
Epoch 01 | loss=5471.60 | val_acc=0.465
Epoch 2 started...
Epoch 02 | loss=5167.70 | val_acc=0.507
Epoch 3 started...
Epoch 03 | loss=5082.60 | val_acc=0.531
Epoch 4 started...
Epoch 04 | loss=5024.28 | val_acc=0.548
Epoch 5 started...
Epoch 05 | loss=4977.34 | val_acc=0.565
Epoch 6 started...
Epoch 06 | loss=4940.95 | val_acc=0.571
Epoch 7 started...
Epoch 07 | loss=4914.05 | val_acc=0.585
Epoch 8 started...
Epoch 08 | loss=4890.91 | val_acc=0.590
Epoch 9 started...
Epoch 09 | loss=4871.78 | val_acc=0.592
Epoch 10 started...
Epoch 10 | loss=4851.34 | val_acc=0.601
Epoch 11 started...
Epoch 11 | loss=4834.03 | val_acc=0.607
Epoch 12 started...
Epoch 12 | loss=4821.98 | val_acc=0.611
Epoch 13 started...
Epoch 13 | loss=4810.40 | val_acc=0.614
Epoch 14 started...
Epoch 14 | loss=4798.40 | val_acc=0.615
Epoch 15 started...
Epoch 15 | loss=4789.21 | val_acc=0.620
Epoch 16 started...
Epoch 16 | loss=4780.06 | val_acc=0.621
Epoch 17 started...
Epoch 17 | loss=4773.29 | val

In [ ]:
trans.fit(X_vit, y, epochs=35, batch_size=128, validation_split=0.15)

In [ ]:
trans.save("transformer_connect4.keras")

In [ ]:
from google.colab import files
files.download("cnn_connect4.pt")
files.download("transformer_connect4.keras")